In [1]:
#| default_exp skill

## Installing the skill

Install the notebook workflow instructions and their reference notes for agents.

The repository also ships the instructions that teach an agent how to use these tools. This notebook installs the bundled `SKILL.md` and references into a local Codex or Claude skills directory so the notebook-first workflow can travel with the package.

The skill files are packaging for agent behavior, not another implementation path. They point agents back to the same notebook-aware tools in this repository, so users get consistent reads, writes, execution, and review whether they invoke nbskill through Python, the CLI, or MCP.

```python
build_skill_from_readme("README.md", "nbskill/SKILL.md")
install_nbskill(dest="~/.codex/skills/jupyter-notebooks")
```

In [2]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from nbskill.skill import build_skill_from_readme as _example_build_skill_from_readme
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

ImportError: cannot import name 'demo_path' from 'nbskill.foundation' (/Users/macbook/Projects/nbskill/nbskill/foundation.py)

In [ ]:
root = demo_path("06_skill_example")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    readme.write_text("before\n<!-- nbskill-skill:start -->\n# Skill Body\nUse read_nb first.\n<!-- nbskill-skill:end -->\nafter\n", encoding="utf-8")
    _example_build_skill_from_readme(str(readme), str(out))
    print("built:", out.name)
    print(out.read_text(encoding="utf-8").splitlines()[-2])
finally:
    remove_demo_path(root)

Built nbs/data/06_skill_example/SKILL.md from nbs/data/06_skill_example/README.md
built: SKILL.md
# Skill Body


In [ ]:
#| export
from importlib.resources import files
from pathlib import Path

from fastcore.script import call_parse

from nbskill.foundation import cli_return, tracked_call

In [ ]:
#| export
_SKILL_START = "<!-- nbskill-skill:start -->"
_SKILL_END = "<!-- nbskill-skill:end -->"
_SKILL_FRONTMATTER = """---
name: jupyter-notebooks
description: Work notebook-first in nbdev projects with nbskill MCP tools for reading, writing, updating, and executing notebooks without raw JSON.
---"""


def _marked_readme_section(text, start=_SKILL_START, end=_SKILL_END):
    try:
        body = text.split(start, 1)[1].split(end, 1)[0]
    except IndexError as exc:
        raise ValueError(f"README is missing {start!r} and {end!r} markers") from exc
    return body.strip()


def _skill_frontmatter(out_path):
    path = Path(out_path)
    if path.exists():
        text = path.read_text(encoding="utf-8")
        if text.startswith("---\n"):
            parts = text.split("\n---", 1)
            if len(parts) == 2: return f"---{parts[0][3:]}\n---"
    return _SKILL_FRONTMATTER


@call_parse
@tracked_call
def build_skill_from_readme(
    readme_path: str = "README.md",  # README containing the marked skill section
    out_path: str = "nbskill/SKILL.md",  # Skill file to write
):
    "Build SKILL.md from the marked section of README.md."
    out = Path(out_path)
    body = _marked_readme_section(Path(readme_path).read_text(encoding="utf-8"))
    text = f"{_skill_frontmatter(out)}\n\n{body}\n"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(text, encoding="utf-8")
    print(f"Built {out} from {readme_path}")
    return cli_return(out)

### Installing agent instructions

The installer resolves the target skills directory, copies the packaged `SKILL.md`, and includes reference files. That keeps the project documentation and the agent workflow in sync with the package version.

In [ ]:
#| export
@call_parse
@tracked_call
def install_nbskill(
    target: str = "codex",  # codex, claude, both, or custom when skills_dir is set
    skills_dir: str | None = None,  # Parent skills directory; skill is installed below jupyter-notebooks
    skill_name: str = "jupyter-notebooks",  # Skill folder name
    overwrite: bool = True,  # Overwrite an existing SKILL.md
):
    "Install the bundled SKILL.md into a Codex or Claude Code skills directory."
    target = target.lower()
    if skills_dir: roots = [Path(skills_dir).expanduser()]
    elif target == "codex": roots = [Path.home() / ".codex" / "skills"]
    elif target in {"claude", "claude-code", "claude_code"}: roots = [Path.home() / ".claude" / "skills"]
    elif target == "both": roots = [Path.home() / ".codex" / "skills", Path.home() / ".claude" / "skills"]
    else: raise ValueError("target must be codex, claude, both, or use skills_dir")
    package = files("nbskill")
    skill_text = package.joinpath("SKILL.md").read_text(encoding="utf-8")
    references = package.joinpath("references")
    installed = []
    for root in roots:
        dst_dir = root / skill_name
        dst = dst_dir / "SKILL.md"
        if dst.exists() and not overwrite: raise FileExistsError(dst)
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst.write_text(skill_text, encoding="utf-8")
        if references.is_dir():
            ref_dir = dst_dir / "references"
            ref_dir.mkdir(exist_ok=True)
            for ref in references.iterdir():
                if ref.is_file():
                    (ref_dir / ref.name).write_text(ref.read_text(encoding="utf-8"), encoding="utf-8")
        installed.append(dst)
    msg = "\n".join(f"Installed {path}" for path in installed)
    print(msg)
    return cli_return(installed)

In [ ]:
install_root = demo_path("06_skill_install")
try:
    install_nbskill(skills_dir=str(install_root))
    skill_dir = install_root / "jupyter-notebooks"
    assert (skill_dir / "SKILL.md").exists()
    assert (skill_dir / "references" / "mcp-tools.md").exists()
    assert (skill_dir / "references" / "cli-fallbacks.md").exists()
    assert (skill_dir / "references" / "conversion.md").exists()
    assert (skill_dir / "references" / "extended-tools.md").exists()
finally:
    remove_demo_path(install_root)

root = demo_path("06_skill_build")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    readme.write_text(
        """before
<!-- nbskill-skill:start -->
# Skill Body
Use notebooks.
<!-- nbskill-skill:end -->
after
""",
        encoding="utf-8",
    )
    build_skill_from_readme(str(readme), str(out))
    text = out.read_text(encoding="utf-8")
    assert text.startswith("---\nname: jupyter-notebooks")
    assert "# Skill Body" in text
    assert "before" not in text and "after" not in text
    out.write_text("---\nname: custom-skill\ndescription: Keep me.\n---\nold\n", encoding="utf-8")
    build_skill_from_readme(str(readme), str(out))
    text = out.read_text(encoding="utf-8")
    assert text.startswith("---\nname: custom-skill")
    assert "# Skill Body" in text and "old" not in text
finally:
    remove_demo_path(root)